# EcoSpold workflow without a Brightway project

This notebook exercises the workflow introduced in `premise` where you can:

- read ecoinvent directly from EcoSpold files,
- run `NewDatabase(...)` without creating or activating a Brightway project,
- export to a non-Brightway format such as sparse matrices, Simapro CSV, or OpenLCA CSV.

Brightway is still an installed dependency of `premise`, but this workflow does not call `bw2data.projects.set_current(...)` and does not require `bw2io.create_default_biosphere3()`.


## Prerequisites

Before running the notebook, adjust the configuration in the next cell:

- `ECOSPOLD_DIR` must point to your local ecoinvent `datasets/` directory.
- `SOURCE_VERSION` must match the ecoinvent version of those EcoSpold files.
- `IAM_KEY` is needed only if you use standard encrypted IAM scenarios shipped with `premise`.
- `TRANSFORMATIONS` can be set to `None` for a full `ndb.update()` run, or to a short list for a lighter smoke test.


In [5]:
from pathlib import Path
import os

from premise import NewDatabase

# --- user configuration -------------------------------------------------
ECOSPOLD_DIR = Path("/Users/romain/Documents/ecoinvent 3.12_cutoff_ecoSpold02/datasets")
SOURCE_VERSION = "3.12"

SCENARIOS = [
    {"model": "image", "pathway": "SSP2-M", "year": 2030},
]

# Set to None to run the full workflow with ndb.update().
# TRANSFORMATIONS = ["electricity"]
TRANSFORMATIONS = None

# Export options: "matrices", "simapro", or "olca"
EXPORT_KIND = "simapro"
EXPORT_DIR = Path.cwd() / "export" / "ecospold_without_brightway_project"

# Optional: exported IAM key from your shell environment.
IAM_KEY = "tUePmX_S5B8ieZkkM7WUU2CnO8SmShwmAeWK9x2rTFo="

assert ECOSPOLD_DIR.is_dir(), f"EcoSpold directory not found: {ECOSPOLD_DIR}"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"EcoSpold source: {ECOSPOLD_DIR}")
print(f"Source version: {SOURCE_VERSION}")
print(f"Export kind: {EXPORT_KIND}")
print(f"Transformations: {TRANSFORMATIONS}")


EcoSpold source: /Users/romain/Documents/ecoinvent 3.12_cutoff_ecoSpold02/datasets
Source version: 3.12
Export kind: simapro
Transformations: None


## Build the database from EcoSpold

The key point here is that there is no Brightway project setup before calling `NewDatabase(...)`.


In [6]:
ndb = NewDatabase(
    scenarios=SCENARIOS,
    source_type="ecospold",
    source_file_path=str(ECOSPOLD_DIR),
    source_version=SOURCE_VERSION,
    key=IAM_KEY,
)

if TRANSFORMATIONS is None:
    ndb.update()
else:
    ndb.update(TRANSFORMATIONS)

print("Workflow completed without creating a Brightway project.")


premise v.(2, 3, 8)
+------------------------------------------------------------------+
| Warning                                                          |
+------------------------------------------------------------------+
| Because some of the scenarios can yield LCI databases            |
| containing net negative emission technologies (NET),             |
| it is advised to account for biogenic CO2 flows when calculating |
| Global Warming potential indicators.                             |
| `premise_gwp` provides characterization factors for such flows.  |
| It also provides factors for hydrogen emissions to air.          |
|                                                                  |
| Within your Brightway project:                                   |
| from premise_gwp import add_premise_gwp                          |
| add_premise_gwp()                                                |
+------------------------------------------------------------------+
+-------------

Processing scenarios for all sectors:   0%|     | 0/1 [00:00<?, ?it/s]

No DAC energy mix data available -- skipping
No EWR energy mix data available -- skipping
No two-wheeler fleet scenario data available -- skipping


Processing scenarios for all sectors: 100%|█| 1/1 [02:37<00:00, 157.71

Done!

Workflow completed without creating a Brightway project.


## Export without Brightway

Pick one of the supported non-Brightway export paths.


In [7]:
if EXPORT_KIND == "matrices":
    out = EXPORT_DIR / "matrices"
    ndb.write_db_to_matrices(filepath=str(out))
elif EXPORT_KIND == "simapro":
    out = EXPORT_DIR / "simapro"
    ndb.write_db_to_simapro(filepath=str(out))
elif EXPORT_KIND == "olca":
    out = EXPORT_DIR / "olca"
    ndb.write_db_to_olca(filepath=str(out))
else:
    raise ValueError(f"Unsupported EXPORT_KIND: {EXPORT_KIND}")

print(f"Export written under: {out}")


Write Simapro import file(s).
Running all checks...
Minor anomalies found: check the change report.
The following exchanges have not been used in the Simapro export:
+---------------------------------------+---------+----------------------------------+----------+
|                  Name                 | Product |            Categories            | Location |
+---------------------------------------+---------+----------------------------------+----------+
|      Non-hazardous waste disposed     |         | ('inventory indicator', 'waste') |   None   |
|        Hazardous waste disposed       |         | ('inventory indicator', 'waste') |   None   |
|   Organic carbon, placed in landfill  |         | ('inventory indicator', 'waste') |   None   |
| Waste mass, total, placed in landfill |         | ('inventory indicator', 'waste') |   None   |
+---------------------------------------+---------+----------------------------------+----------+
173 unmatched flow categories. Check unlinked.log.